In [2]:
# 1. 라이브러리
from pathlib import Path
import json
import sqlite3

import pandas as pd
from tqdm.auto import tqdm

c:\Users\playdata2\SKN34-2nd-5Team\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 2. 경로 설정
# 현재 실행 위치에서 프로젝트 최상위 폴더 탐색
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "data" / "raw").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "data/raw 폴더를 찾을 수 없습니다."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_JSON = RAW_DIR / "yelp_academic_dataset_review.json"
USER_JSON = RAW_DIR / "yelp_academic_dataset_user.json"
BUSINESS_JSON = RAW_DIR / "yelp_academic_dataset_business.json"

print("프로젝트 루트:", PROJECT_ROOT)
print("Review:", REVIEW_JSON.exists())
print("User:", USER_JSON.exists())
print("Business:", BUSINESS_JSON.exists())

프로젝트 루트: C:\Users\playdata2\SKN34-2nd-5Team
Review: True
User: True
Business: True


In [4]:
# 3. User·Business 기본키 검증 함수
def load_and_validate_ids(
    file_path,
    key_column,
    expected_rows=None
):
    """
    JSON Lines 파일을 한 줄씩 읽으면서
    기본키의 고유값, 중복, 결측을 검사한다.

    반환값
    -------
    unique_ids : set
        고유한 ID 집합
    result : dict
        행 수, 고유값 수, 중복 수, 결측 수
    """
    unique_ids = set()

    total_rows = 0
    duplicate_keys = 0
    null_keys = 0

    with file_path.open(
        mode="r",
        encoding="utf-8"
    ) as file:

        for line in tqdm(
            file,
            total=expected_rows,
            desc=f"{key_column} 검증"
        ):
            if not line.strip():
                continue

            record = json.loads(line)
            key_value = record.get(key_column)

            total_rows += 1

            # None 또는 빈 문자열을 결측으로 처리
            if key_value is None or key_value == "":
                null_keys += 1
                continue

            if key_value in unique_ids:
                duplicate_keys += 1
            else:
                unique_ids.add(key_value)

    result = {
        "total_rows": total_rows,
        "unique_keys": len(unique_ids),
        "duplicate_keys": duplicate_keys,
        "null_keys": null_keys
    }

    return unique_ids, result

In [5]:
# 4. User 기본키 검증
user_ids, user_key_result = load_and_validate_ids(
    file_path=USER_JSON,
    key_column="user_id",
    expected_rows=1_987_897
)

user_key_result

user_id 검증: 100%|██████████| 1987897/1987897 [00:29<00:00, 68410.69it/s] 


{'total_rows': 1987897,
 'unique_keys': 1987897,
 'duplicate_keys': 0,
 'null_keys': 0}

In [6]:
# 5. Business 기본키 검증
business_ids, business_key_result = load_and_validate_ids(
    file_path=BUSINESS_JSON,
    key_column="business_id",
    expected_rows=150_346
)

business_key_result

business_id 검증: 100%|██████████| 150346/150346 [00:01<00:00, 79017.56it/s]


{'total_rows': 150346,
 'unique_keys': 150346,
 'duplicate_keys': 0,
 'null_keys': 0}

In [7]:
# 6. Review 기본키 및 연결률 검증
VALIDATION_DB = INTERIM_DIR / "key_validation.sqlite"

sqlite_con = sqlite3.connect(VALIDATION_DB)

# 검증 속도를 높이기 위한 설정
sqlite_con.execute("PRAGMA synchronous = OFF")
sqlite_con.execute("PRAGMA journal_mode = MEMORY")

# 노트북을 다시 실행해도 동일한 결과가 나오도록 테이블 재생성
sqlite_con.execute("DROP TABLE IF EXISTS review_ids")

sqlite_con.execute(
    """
    CREATE TABLE review_ids (
        review_id TEXT PRIMARY KEY
    )
    """
)

sqlite_con.commit()

print("SQLite 검증 테이블 생성 완료")

SQLite 검증 테이블 생성 완료


In [8]:
# Review 스트리밍 검증 함수
def validate_review_keys_and_joins(
    file_path,
    sqlite_connection,
    valid_user_ids,
    valid_business_ids,
    expected_rows=None,
    batch_size=50_000
):
    """
    Review JSON을 한 줄씩 읽으며 다음을 검증한다.

    1. review_id 중복·결측
    2. Review → User 연결 여부
    3. Review → Business 연결 여부
    """
    total_rows = 0

    review_id_nulls = 0
    user_id_nulls = 0
    business_id_nulls = 0

    matched_user_rows = 0
    matched_business_rows = 0

    non_null_review_ids = 0
    inserted_review_ids = 0

    review_id_batch = []

    def insert_batch(batch):
        """
        Review ID 묶음을 SQLite에 저장하고
        실제로 새롭게 저장된 ID 수를 반환한다.
        """
        if not batch:
            return 0

        before_changes = sqlite_connection.total_changes

        sqlite_connection.executemany(
            """
            INSERT OR IGNORE INTO review_ids (review_id)
            VALUES (?)
            """,
            batch
        )

        return (
            sqlite_connection.total_changes
            - before_changes
        )

    with file_path.open(
        mode="r",
        encoding="utf-8"
    ) as file:

        for line in tqdm(
            file,
            total=expected_rows,
            desc="Review 검증"
        ):
            if not line.strip():
                continue

            record = json.loads(line)

            review_id = record.get("review_id")
            user_id = record.get("user_id")
            business_id = record.get("business_id")

            total_rows += 1

            # Review 기본키 검사
            if review_id is None or review_id == "":
                review_id_nulls += 1
            else:
                non_null_review_ids += 1
                review_id_batch.append((review_id,))

            # Review → User 연결 검사
            if user_id is None or user_id == "":
                user_id_nulls += 1
            elif user_id in valid_user_ids:
                matched_user_rows += 1

            # Review → Business 연결 검사
            if business_id is None or business_id == "":
                business_id_nulls += 1
            elif business_id in valid_business_ids:
                matched_business_rows += 1

            # 일정 개수마다 SQLite에 저장
            if len(review_id_batch) >= batch_size:
                inserted_review_ids += insert_batch(
                    review_id_batch
                )

                review_id_batch = []

        # 마지막 남은 데이터 저장
        inserted_review_ids += insert_batch(
            review_id_batch
        )

    sqlite_connection.commit()

    duplicate_review_ids = (
        non_null_review_ids
        - inserted_review_ids
    )

    result = {
        "total_rows": total_rows,
        "unique_review_ids": inserted_review_ids,
        "duplicate_review_ids": duplicate_review_ids,
        "review_id_nulls": review_id_nulls,
        "user_id_nulls": user_id_nulls,
        "business_id_nulls": business_id_nulls,
        "matched_user_rows": matched_user_rows,
        "unmatched_user_rows": (
            total_rows
            - matched_user_rows
            - user_id_nulls
        ),
        "matched_business_rows": matched_business_rows,
        "unmatched_business_rows": (
            total_rows
            - matched_business_rows
            - business_id_nulls
        )
    }

    return result

In [9]:
# 7. Review 검증 실행
review_validation_result = validate_review_keys_and_joins(
    file_path=REVIEW_JSON,
    sqlite_connection=sqlite_con,
    valid_user_ids=user_ids,
    valid_business_ids=business_ids,
    expected_rows=6_990_280
)

review_validation_result


Review 검증: 100%|██████████| 6990280/6990280 [04:20<00:00, 26841.24it/s]


{'total_rows': 6990280,
 'unique_review_ids': 6990280,
 'duplicate_review_ids': 0,
 'review_id_nulls': 0,
 'user_id_nulls': 0,
 'business_id_nulls': 0,
 'matched_user_rows': 6990247,
 'unmatched_user_rows': 33,
 'matched_business_rows': 6990280,
 'unmatched_business_rows': 0}

In [10]:
# 8. 기본키 결과 정리
key_validation_df = pd.DataFrame(
    [
        {
            "dataset": "Review",
            "key_column": "review_id",
            "total_rows": review_validation_result[
                "total_rows"
            ],
            "unique_keys": review_validation_result[
                "unique_review_ids"
            ],
            "duplicate_keys": review_validation_result[
                "duplicate_review_ids"
            ],
            "null_keys": review_validation_result[
                "review_id_nulls"
            ]
        },
        {
            "dataset": "User",
            "key_column": "user_id",
            **user_key_result
        },
        {
            "dataset": "Business",
            "key_column": "business_id",
            **business_key_result
        }
    ]
)

key_validation_df

,dataset,key_column,total_rows,unique_keys,duplicate_keys,null_keys
0,Review,review_id,6990280,6990280,0,0
1,User,user_id,1987897,1987897,0,0
2,Business,business_id,150346,150346,0,0


In [11]:
# 9. 연결률 결과 정리
total_review_rows = review_validation_result[
    "total_rows"
]

join_validation_df = pd.DataFrame(
    [
        {
            "relationship": "Review → User",
            "total_rows": total_review_rows,
            "matched_rows": review_validation_result[
                "matched_user_rows"
            ],
            "unmatched_rows": review_validation_result[
                "unmatched_user_rows"
            ],
            "null_key_rows": review_validation_result[
                "user_id_nulls"
            ],
            "match_rate_pct": round(
                review_validation_result[
                    "matched_user_rows"
                ]
                / total_review_rows
                * 100,
                6
            )
        },
        {
            "relationship": "Review → Business",
            "total_rows": total_review_rows,
            "matched_rows": review_validation_result[
                "matched_business_rows"
            ],
            "unmatched_rows": review_validation_result[
                "unmatched_business_rows"
            ],
            "null_key_rows": review_validation_result[
                "business_id_nulls"
            ],
            "match_rate_pct": round(
                review_validation_result[
                    "matched_business_rows"
                ]
                / total_review_rows
                * 100,
                6
            )
        }
    ]
)

join_validation_df

,relationship,total_rows,matched_rows,unmatched_rows,null_key_rows,match_rate_pct
0,Review → User,6990280,6990247,33,0,99.999528
1,Review → Business,6990280,6990280,0,0,100.000000


In [12]:
# 10. 결과 저장
key_validation_df.to_csv(
    REPORT_TABLE_DIR / "key_validation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

join_validation_df.to_csv(
    REPORT_TABLE_DIR / "join_match_rates.csv",
    index=False,
    encoding="utf-8-sig"
)

print("검증 결과 저장 완료")

검증 결과 저장 완료


In [13]:
sqlite_con.close()

print("SQLite 연결 종료")

SQLite 연결 종료
